# 02 — Preprocessing — CIC-IDS-2017 — SVM-RFE study

Input: `data/raw_unified.parquet` (notebook 01). Output: `data/clean.parquet` + fitted
label encoder. Every dropped row is counted and saved to `data/preprocess_stats.json`.
Scaling and class balancing are deliberately left to the in-pipeline stages (fit on each
scheme's training portion only — no leakage), exactly as in the base study.

In [1]:
DATASET = "cic2017"
FT_ROOT = "ft40_rfe"

In [2]:
import sys
from pathlib import Path

def _find_nb_common():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for c in (cand, cand / "notebooks"):
            if (c / "nb_common.py").exists():
                return c
    raise FileNotFoundError("nb_common.py not found relative to " + str(Path.cwd()))
sys.path.insert(0, str(_find_nb_common()))

import pandas as pd

import nb_common as nbc
from ids import config
from ids.schema import CANONICAL_COLUMNS, LABEL_COLUMN
from ids.preprocess.cleaning import clean
from ids.preprocess.encoding import fit_label_encoder, save_label_encoder, transform_labels

P = nbc.ft_paths(DATASET, FT_ROOT)
log = nbc.setup_logging(P.logs / "02_preprocess.log", "nb02")

with nbc.timed(log, "load raw_unified.parquet"):
    raw = pd.read_parquet(P.data / "raw_unified.parquet")
log.info("input: %d rows x %d cols", *raw.shape)

2026-07-06 11:18:04,094 INFO nb02: START load raw_unified.parquet
2026-07-06 11:18:05,778 INFO nb02: END   load raw_unified.parquet (1.7s)
2026-07-06 11:18:05,785 INFO nb02: input: 2830743 rows x 78 cols


## 1. Clean (inf→NaN, drop NaN rows, drop duplicates)

In [3]:
with nbc.timed(log, "clean (inf->NaN, drop NaN rows, drop duplicates)"):
    clean_df, stats = clean(raw)
stats["pct_rows_dropped"] = round(100 * (1 - stats["output_rows"] / stats["input_rows"]), 3)
nbc.save_json(stats, P.data / "preprocess_stats.json")
log.info("cleaning: %s", stats)
stats

2026-07-06 11:18:05,808 INFO nb02: START clean (inf->NaN, drop NaN rows, drop duplicates)
2026-07-06 11:19:00,851 INFO ids.preprocess.cleaning: cleaning stats: {'input_rows': 2830743, 'replaced_inf': 4376, 'dropped_nan': 2867, 'dropped_duplicates': 307078, 'output_rows': 2520798}
2026-07-06 11:19:06,422 INFO nb02: END   clean (inf->NaN, drop NaN rows, drop duplicates) (60.6s)
2026-07-06 11:19:06,641 INFO nb02: cleaning: {'input_rows': 2830743, 'replaced_inf': 4376, 'dropped_nan': 2867, 'dropped_duplicates': 307078, 'output_rows': 2520798, 'pct_rows_dropped': 10.949}


{'input_rows': 2830743,
 'replaced_inf': 4376,
 'dropped_nan': 2867,
 'dropped_duplicates': 307078,
 'output_rows': 2520798,
 'pct_rows_dropped': 10.949}

### First and last 10 rows of the cleaned dataset

Quick visual check of the table the models will see (all 77 features + label).
Also saved to `data/clean_head_tail_preview.csv`.

In [4]:
preview = pd.concat([clean_df.head(10), clean_df.tail(10)])
preview.to_csv(P.data / "clean_head_tail_preview.csv")
log.info("saved clean_head_tail_preview.csv (%d rows)", len(preview))
with pd.option_context("display.max_columns", None):
    display(clean_df.head(10))
    display(clean_df.tail(10))

2026-07-06 11:19:08,028 INFO nb02: saved clean_head_tail_preview.csv (20 rows)


,dst_port,flow_duration,tot_fwd_pkts,tot_bwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,fwd_pkt_len_max,fwd_pkt_len_min,fwd_pkt_len_mean,fwd_pkt_len_std,bwd_pkt_len_max,bwd_pkt_len_min,bwd_pkt_len_mean,bwd_pkt_len_std,flow_byts_s,flow_pkts_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_tot,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_tot,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_len,bwd_header_len,fwd_pkts_s,bwd_pkts_s,pkt_len_min,pkt_len_max,pkt_len_mean,pkt_len_std,pkt_len_var,fin_flag_cnt,syn_flag_cnt,rst_flag_cnt,psh_flag_cnt,ack_flag_cnt,urg_flag_cnt,cwe_flag_count,ece_flag_cnt,down_up_ratio,pkt_size_avg,fwd_seg_size_avg,bwd_seg_size_avg,fwd_byts_b_avg,fwd_pkts_b_avg,fwd_blk_rate_avg,bwd_byts_b_avg,bwd_pkts_b_avg,bwd_blk_rate_avg,subflow_fwd_pkts,subflow_fwd_byts,subflow_bwd_pkts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,fwd_act_data_pkts,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865,3,2,0,12,0,6,6,6.0,0.00000,0,0,0.0,0.0,4.000000e+06,666666.666700,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,0,0,0,0,0,0,2,12,0,0,33,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,55054,109,1,1,6,6,6,6,6.0,0.00000,6,6,6.0,0.0,1.100917e+05,18348.623850,109.0,0.0,109,109,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,9174.311927,9174.311927,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,55055,52,1,1,6,6,6,6,6.0,0.00000,6,6,6.0,0.0,2.307692e+05,38461.538460,52.0,0.0,52,52,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,19230.769230,19230.769230,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,46236,34,1,1,6,6,6,6,6.0,0.00000,6,6,6.0,0.0,3.529412e+05,58823.529410,34.0,0.0,34,34,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,29411.764710,29411.764710,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,0,0,0,0,0,0,1,6,1,6,31,329,0,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,54863,3,2,0,12,0,6,6,6.0,0.00000,0,0,0.0,0.0,4.000000e+06,666666.666700,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
5,54871,1022,2,0,12,0,6,6,6.0,0.00000,0,0,0.0,0.0,1.174168e+04,1956.947162,1022.0,0.0,1022,1022,1022,1022.0,0.0,1022,1022,0,0.0,0.0,0,0,0,0,0,0,40,0,1956.947162,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
6,54925,4,2,0,12,0,6,6,6.0,0.00000,0,0,0.0,0.0,3.000000e+06,500000.000000,4.0,0.0,4,4,4,4.0,0.0,4,4,0,0.0,0.0,0,0,0,0,0,0,40,0,500000.000000,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
7,54925,42,1,1,6,6,6,6,6.0,0.00000,6,6,6.0,0.0,2.857143e+05,47619.047620,42.0,0.0,42,42,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,23809.523810,23809.523810,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,1,9.0,6.0,6.0,0,0,0,0,0,0,1,6,1,6,32,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
8,9282,4,2,0,12,0,6,6,6.0,0.00000,0,0,0.0,0.0,3.000000e+06,500000.000000,4.0,0.0,4,4,4,4.0,0.0,4,4,0,0.0,0.0,0,0,0,0,0,0,40,0,500000.000000,0.000000,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
9,55153,4,2,0,37,0,31,6,18.5,17.67767,0,0,0.0,0.0,9.250000e+06,500000.000000,4.0,0.0,4,4,4,4.0,0.0,4,4,0,0.0,0.0,0,0,1,0,0,0,40,0,500000.000000,0.000000,6,31,22.666667,14.433757,208.333333,0,1,0,0,1,0,0,0,0,34.0,18.5,0.0,0,0,0,0,0,0,2,37,0,0,946,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


,dst_port,flow_duration,tot_fwd_pkts,tot_bwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,fwd_pkt_len_max,fwd_pkt_len_min,fwd_pkt_len_mean,fwd_pkt_len_std,bwd_pkt_len_max,bwd_pkt_len_min,bwd_pkt_len_mean,bwd_pkt_len_std,flow_byts_s,flow_pkts_s,flow_iat_mean,flow_iat_std,flow_iat_max,flow_iat_min,fwd_iat_tot,fwd_iat_mean,fwd_iat_std,fwd_iat_max,fwd_iat_min,bwd_iat_tot,bwd_iat_mean,bwd_iat_std,bwd_iat_max,bwd_iat_min,fwd_psh_flags,bwd_psh_flags,fwd_urg_flags,bwd_urg_flags,fwd_header_len,bwd_header_len,fwd_pkts_s,bwd_pkts_s,pkt_len_min,pkt_len_max,pkt_len_mean,pkt_len_std,pkt_len_var,fin_flag_cnt,syn_flag_cnt,rst_flag_cnt,psh_flag_cnt,ack_flag_cnt,urg_flag_cnt,cwe_flag_count,ece_flag_cnt,down_up_ratio,pkt_size_avg,fwd_seg_size_avg,bwd_seg_size_avg,fwd_byts_b_avg,fwd_pkts_b_avg,fwd_blk_rate_avg,bwd_byts_b_avg,bwd_pkts_b_avg,bwd_blk_rate_avg,subflow_fwd_pkts,subflow_fwd_byts,subflow_bwd_pkts,subflow_bwd_byts,init_fwd_win_byts,init_bwd_win_byts,fwd_act_data_pkts,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
2520788,53,247,2,2,74,426,37,37,37.0,0.000000,213,213,213.0,0.00000,2.024291e+06,16194.331980,8.233333e+01,1.000217e+02,195,4,48,4.800000e+01,0.000000e+00,48,48,4,4.0,0.000,4,4,0,0,0,0,64,64,8097.165992,8097.165992,37,213,107.400000,96.399170,9292.800000,0,0,0,0,0,0,0,0,1,134.250000,37.0,213.0,0,0,0,0,0,0,2,74,2,426,-1,-1,1,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2520789,35450,33,1,1,0,0,0,0,0.0,0.000000,0,0,0.0,0.00000,0.000000e+00,60606.060610,3.300000e+01,0.000000e+00,33,33,0,0.000000e+00,0.000000e+00,0,0,0,0.0,0.000,0,0,0,0,0,0,32,32,30303.030300,30303.030300,0,0,0.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,0.000000,0.0,0.0,0,0,0,0,0,0,1,0,1,0,283,229,0,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2520790,53,46982,2,2,58,446,29,29,29.0,0.000000,223,223,223.0,0.00000,1.072751e+04,85.138989,1.566067e+04,2.712074e+04,46977,1,1,1.000000e+00,0.000000e+00,1,1,4,4.0,0.000,4,4,0,0,0,0,40,40,42.569495,42.569495,29,223,106.600000,106.258176,11290.800000,0,0,0,0,0,0,0,0,1,133.250000,29.0,223.0,0,0,0,0,0,0,2,58,2,446,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2520791,80,11512230,5,5,672,3525,672,0,134.4,300.527536,2077,0,705.0,990.64474,3.645688e+02,0.868641,1.279137e+06,2.565060e+06,6506213,4,6509115,1.627279e+06,3.253892e+06,6508117,4,11500000,2878045.5,3378842.065,6506213,49,0,0,0,0,168,168,0.434321,0.434321,0,2077,381.545455,725.318601,526087.072700,0,0,0,1,0,0,0,0,1,419.700000,134.4,705.0,0,0,0,0,0,0,5,672,5,3525,29200,237,1,32,821.0,0.0,821,821,6506213.0,0.0,6506213,6506213,DoS GoldenEye
2520792,49346,145,1,1,0,0,0,0,0.0,0.000000,0,0,0.0,0.00000,0.000000e+00,13793.103450,1.450000e+02,0.000000e+00,145,145,0,0.000000e+00,0.000000e+00,0,0,0,0.0,0.000,0,0,0,0,0,0,32,32,6896.551724,6896.551724,0,0,0.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,0.000000,0.0,0.0,0,0,0,0,0,0,1,0,1,0,3,229,0,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2520793,53,32215,4,2,112,152,28,28,28.0,0.000000,76,76,76.0,0.00000,8.194940e+03,186.248642,6.443000e+03,1.361758e+04,30780,3,30832,1.027733e+04,1.775584e+04,30780,4,3,3.0,0.000,3,3,0,0,0,0,80,64,124.165761,62.082881,28,76,41.714286,23.421602,548.571429,0,0,0,0,0,0,0,0,0,48.666667,28.0,76.0,0,0,0,0,0,0,4,112,2,152,-1,-1,3,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2520794,53,324,2,2,84,362,42,42,42.0,0.000000,181,181,181.0,0.00000,1.376543e+06,12345.679010,1.080000e+02,1.835974e+02,320,2,2,2.000000e+00,0.000000e+00,2,2,2,2.0,0.000,2,2,0,0,0,0,40,40,6172.839506,6172.839506,42,181,97.600000,76.133435,5796.300000,0,0,0,0,0,0,0,0,1,122.000000,42.0,181.0,0,0,0,0,0,0,2,84,2,362,-1,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2520795,58030,82,2,1,31,6,31,0,15.5,21.920310,6,6,6.0,0.00000,4.512195e+05,36585.365850,4.100000e+01,5.232590e+01,78,4,4,4.000000e+00,0.000000e+00,4,4,0,0.0,0.000,0,0,1,0,0,0,64,20,24390.243900,12195.121950,0,31,17.000000,16.350331,267.333333,0,1,0,0,1,0,0,0,0,22.666667,15.5,6.0,0,0,0,0,0,0,2,31,1,6,1006,0,0,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2520796,53,1048635,6,2,192,256,32,32,32.0,0.0000

## 2. Encode labels + save the clean modeling table

In [5]:
encoder = fit_label_encoder(clean_df[LABEL_COLUMN])
save_label_encoder(encoder, P.data / "label_encoder.joblib")
y = transform_labels(encoder, clean_df[LABEL_COLUMN])
mapping = {cls: int(i) for i, cls in enumerate(encoder.classes_)}
nbc.save_json(mapping, P.data / "label_mapping.json")

out = clean_df[CANONICAL_COLUMNS].copy()
out["y"] = y
out.to_parquet(P.data / "clean.parquet", index=False)

final_dist = (clean_df[LABEL_COLUMN].value_counts()
              .rename_axis("label").reset_index(name="count"))
final_dist["pct"] = (100 * final_dist["count"] / len(clean_df)).round(4)
final_dist.to_csv(P.data / "final_class_distribution.csv", index=False)
log.info("saved clean.parquet: %d rows x %d features (+y) | %d classes",
         len(out), len(CANONICAL_COLUMNS), final_dist.shape[0])
final_dist

2026-07-06 11:19:19,926 INFO nb02: saved clean.parquet: 2520798 rows x 77 features (+y) | 15 classes


,label,count,pct
0,Benign,2095057,83.1109
1,DoS Hulk,172846,6.8568
2,DDoS,128014,5.0783
3,PortScan,90694,3.5978
4,DoS GoldenEye,10286,0.4080
5,FTP-Patator,5931,0.2353
6,DoS slowloris,5385,0.2136
7,DoS Slowhttptest,5228,0.2074
8,SSH-Patator,3219,0.1277
9,Bot,1948,0.0773
